In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
# pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기
# ============================================================

df = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'train.parquet')

In [2]:
# ============================================================
# 1. 시계열데이터의 강우사상별 분리(event_id부여)
# ============================================================
# - 전체 데이터의 고유 timestamp 추출 및 시간순 정렬
# - 이전 timestamp와의 시간간격(diff) 계산
# - 1시간 연속 여부를 기준으로 event 경계 생성
# - 강우사상 시간대별 event_id 부여

# 1-1. 전체 timestamp 추출(중복없이), 시간순 정렬
event_time = (
    df[['timestamp']]  # 이중리스트: 나중에 차분 열 추가를 위해 시리즈가 아닌 df로 만들기
    .drop_duplicates() # 중복제거(unique 효과)
    .sort_values('timestamp') # 기본값: 오름차순 정렬
    .reset_index(drop = True) # 기존 인덱스(row_id) 제거
)
# 이 객체는 시간을 어디서 자를지 판단을 위해 임시로 사용하는 별도 객체, 인덱스를 떼도 됨

# 1-2. diff 계산
event_time['event_time_diff'] = (
    event_time['timestamp']
    .diff()
)

# 1-3. diff가 1시간이 아닌 경우를 event 경계로 추출
cond_split_time = (
    event_time['event_time_diff'] != pd.Timedelta(hours=1)
)
# cond_sort_time은 불리언조건문(diff가 1시간이 아닌경우에만 True 반환)
# 이 diff가 1시간이 아닌 위치가 강우사상 시간대 분할지점

# 1-4. 강우사상 시간대별 event_id 부여
event_time['event_id'] = cond_split_time.cumsum()
# cumsum(): 누적합, 불리언시리즈와 함께 쓰면 True가 나올때마다 값이 누적됨(1씩 증)
display(event_time.head(5))

# 시험처가 제시한 강우사상 개수와 맞는지 확인
print(event_time['event_id'].nunique())

,timestamp,event_time_diff,event_id
0,2100-01-01 00:00:00,NaT,1
1,2100-01-01 01:00:00,0 days 01:00:00,1
2,2100-01-21 00:00:00,19 days 23:00:00,2
3,2100-01-21 01:00:00,0 days 01:00:00,2
4,2100-01-21 02:00:00,0 days 01:00:00,2


122


In [3]:
# ============================================================
# 2. event 기준 데이터 분할
# ============================================================

# df에 event_id 컬럼 붙이기
# Series.map()함수: Series를 map()안의 규칙을 적용, 동일한 크기의 Series를 반환
rule1 = event_time.set_index('timestamp')['event_id']  # map() 규칙만들기

# map() 규칙은 자식 테이블의 조인대상 컬럼이 인덱스여야함
print("timestamp를 event_id로 변환한 시리즈")
display(df['timestamp'].map(rule1))  # 이걸 기존 df 옆에 붙여야 함

#print(len(df['timestamp'].map(rule1)))  # 749657개(누락없음)

# 기존 df에 붙이기
df['event_id'] = df['timestamp'].map(rule1)

# 정상적으로 매핑됐는지 확인하기
# print(f"event_id 컬럼의 원소개수(중복제거): {df['event_id'].nunique()}")
# print()
# print(f"결측치 개수는? {df['event_id'].isnull().sum()}")
# print()
# print(
#     f"""<timestamp - event_id 매핑 일관성 확인>
#     timestamp별 event_id의 고유값 개수의 최대값
#     (중복매칭이 됐으면 2 이상이 나옴): {df.groupby('timestamp')['event_id'].nunique().max()}"""
#     )

split_num = 98
train = df[df['event_id'] <= split_num]
val = df[df['event_id'] > split_num]

print("분할 전후 행열개수")
print(f"분할 전: {df.shape}")
print(f"분할 후: (train){train.shape}, (val){val.shape}")

timestamp를 event_id로 변환한 시리즈


row_id
TR_000222241      1
TR_000288748      1
TR_000737576      1
TR_000651952      1
TR_000328380      1
               ... 
TR_000706792    122
TR_000274711    122
TR_000676453    122
TR_000102391    122
TR_000362309    122
Name: timestamp, Length: 749657, dtype: int64

분할 전후 행열개수
분할 전: (749657, 26)
분할 후: (train)(616332, 26), (val)(133325, 26)


In [4]:
# ============================================================
# 3. 종속변수 이상치 행 제거
# ============================================================
# (중요) 이상치 제거: 1-2 이상치탐색에서 수위변동최대값 오류로 결론낸 'TR_000583385' 관측행 제거
train = train.drop(index = 'TR_000583385')
print(f"이상치 행 제거 후 train셋 행 개수: {len(train)}")  # 1개 줄어듬: good!
print()

# val = val.drop(index = 'TR_000583385')  # 없다고 에러남. val에는 없음

# 잘 분할됐는지 확인하기: 잘됨
print("<df, train, val 데이터셋의 event_id 개수>")
print(df['event_id'].nunique(),
      train['event_id'].nunique(),
      val['event_id'].nunique())
print()

print("<df, train, val 데이터셋의 event_id 개수 대비 총 데이터 수: 비율이 일정해야 좋음>")
print(df.shape[0]/df['event_id'].nunique(), train.shape[0]/train['event_id'].nunique(), val.shape[0]/val['event_id'].nunique())

이상치 행 제거 후 train셋 행 개수: 616331

<df, train, val 데이터셋의 event_id 개수>
122 98 24

<df, train, val 데이터셋의 event_id 개수 대비 총 데이터 수: 비율이 일정해야 좋음>
6144.729508196721 6289.091836734694 5555.208333333333


In [5]:
# ============================================================
# 4. 데이터셋 저장하기
# ============================================================

train.to_parquet(PROJECT_ROOT / 'parqueted_data' / 'train_split.parquet')
val.to_parquet(PROJECT_ROOT / 'parqueted_data' / 'val_split.parquet')
